# 📐 Semana 2 · Unidad 1 — Diseño de Algoritmos: Clase Esencial
**Algoritmos y Estructuras de Datos · Universidad de Talca**

---

Notebook de acompañamiento para la clase de 2 horas. Cada sección presenta un paradigma con un problema nuevo.

| # | Paradigma | Problema de la clase | Tiempo |
|---|---|---|---|
| 1 | **Divide y Vencerás** | Potencia rápida xⁿ | ~20 min |
| 2 | **Greedy** | Selección de actividades | ~20 min |
| 3 | **Programación Dinámica** | Cambio de monedas mínimo | ~20 min |
| 4 | **Backtracking** | Coloración de grafos | ~20 min |
| 5 | **Branch & Bound** | TSP en grafo pequeño | ~15 min |
| — | Cierre | Tabla comparativa | ~5 min |

> Los ejercicios al final de cada sección son para **trabajo autónomo**, no se desarrollan en clases.
> Para profundizar en cada paradigma, revisa los notebooks `NB01`–`NB05`.

In [ ]:
#!pip install --upgrade pip
#!pip install numpy
#!pip install matplotlib

In [ ]:
# ── Imports y paleta de colores del curso ───────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time, random, math
from typing import List, Dict, Tuple, Optional

AZUL     = '#2196F3'   # elemento activo
NARANJA  = '#FF9800'   # comparado / candidato
VERDE    = '#4CAF50'   # solución / correcto
ROJO     = '#F44336'   # podado / descartado
MORADO   = '#9C27B0'   # caso base
AZ_CLARO = '#90CAF9'   # visitado
FONDO    = '#FAFAFA'
TEXTO    = '#212121'

print('✓ Listo.')

---
## Sección 1 — Divide y Vencerás
### Problema: Potencia rápida — calcular xⁿ

**Idea:** si n es par, xⁿ = (x^(n/2))² → solo necesitamos calcular la mitad.
Si n es impar, xⁿ = x · x^(n−1). En cada paso el problema se divide a la mitad → **O(log n)**.

binario(100010)=34 ; 34*2 = binario(100010) <<1 = binario(1000100) = 68
34+4= binario(100010) <<2 = binario(10001000) = 136

In [ ]:
def potencia_lenta(x: float, n: int) -> float:
    '''
    Calcula x^n multiplicando x exactamente n veces.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(1)
    '''
    resultado = 1.0
    for _ in range(n): # iteramos n veces
        resultado *= x # multiplicamos x por sí mismo n veces
    return resultado


def potencia_rapida(x: float, n: int) -> float:
    '''
    Calcula x^n usando Divide y Vencerás.
    Caso base: n == 0 → 1.
    Paso D&V:  n par  → (x^(n/2))²
               n impar → x · x^(n-1)

    Complejidad:
        Tiempo: O(log n)
        Espacio: O(log n)  pila de recursión
    '''
    if n == 0:
        return 1.0 # caso base: x^0 = 1
    if n % 2 == 0: # n par
        mitad = potencia_rapida(x, n // 2) # calculamos x^(n/2) una sola vez
        return mitad * mitad          # un solo producto, no dos llamadas
    return x * potencia_rapida(x, n - 1) # una llamada recursiva menos que la versión lenta


# Verificar que dan el mismo resultado
for x, n in [(2, 10), (3, 8), (1.5, 6)]:
    lenta  = potencia_lenta(x, n)
    rapida = potencia_rapida(x, n)
    igual  = '✓' if abs(lenta - rapida) < 1e-9 else '✗'
    print(f'{x}^{n} = {rapida:.2f}  {igual}')

In [ ]:
# ── Árbol de recursión para 2^8 ─────────────────────────────────────────────
#
# Nivel 0:               2^8
# Nivel 1:          2^4       (se usa dos veces)
# Nivel 2:       2^2         (se usa dos veces)
# Nivel 3:    2^1           (se usa dos veces)
# Nivel 4:  2^0  (caso base)

# Nodos: (x, y, etiqueta)
nodos = [
    (4.0, 4.0, '2⁸'),
    (2.0, 3.0, '2⁴'),   (6.0, 3.0, '(×)²'),
    (1.0, 2.0, '2²'),   (3.0, 2.0, '(×)²'),
    (0.5, 1.0, '2¹'),   (1.5, 1.0, '(×)²'),
    (0.25, 0.0, '2⁰'),  (0.75, 0.0, '×2'),
]
# Aristas: (padre_idx, hijo_idx)
aristas = [(0,1),(0,2),(1,3),(1,4),(3,5),(3,6),(5,7),(5,8)]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)
ax.axis('off')
ax.set_title('Árbol de recursión: potencia_rapida(2, 8)', fontsize=12,
             color=TEXTO, fontweight='bold')

for i, j in aristas:
    x0, y0, _ = nodos[i]
    x1, y1, _ = nodos[j]
    ax.plot([x0, x1], [y0, y1], '-', color='#BDBDBD', lw=1.5, zorder=1)

colores_nodo = [AZUL, AZUL, AZ_CLARO, AZUL, AZ_CLARO,
                AZUL, AZ_CLARO, MORADO, AZ_CLARO]
for (x, y, lbl), col in zip(nodos, colores_nodo):
    circ = plt.Circle((x, y), 0.28, color=col, zorder=2)
    ax.add_patch(circ)
    ax.text(x, y, lbl, ha='center', va='center', fontsize=9,
            color='white', fontweight='bold', zorder=3)

leyenda = [
    mpatches.Patch(color=AZUL,   label='Llamada recursiva'),
    mpatches.Patch(color=AZ_CLARO, label='Solo multiplica (no recurre)'),
    mpatches.Patch(color=MORADO, label='Caso base n=0'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)
ax.set_xlim(-0.3, 8.0)
ax.set_ylim(-0.6, 4.6)

# Etiquetas de nivel
for nivel, txt in enumerate(['n=8', 'n=4', 'n=2', 'n=1', 'n=0']):
    ax.text(-0.2, 4.0 - nivel, txt, ha='right', va='center',
            fontsize=8, color='#757575')

plt.tight_layout()
plt.show()

In [ ]:
# ── Comparación de tiempos: O(n) vs O(log n) ────────────────────────────────
N = 10**6

t0 = time.perf_counter()
potencia_lenta(1.000001, N)
t_lenta = time.perf_counter() - t0

t0 = time.perf_counter()
potencia_rapida(1.000001, N)
t_rapida = time.perf_counter() - t0

print(f'n = {N:,}')
print(f'potencia_lenta  : {t_lenta*1000:8.2f} ms   (O(n)     → {N:,} operaciones)')
print(f'potencia_rapida : {t_rapida*1000:8.4f} ms   (O(log n) → {int(math.log2(N)):,} operaciones)')
print(f'\nFactora de aceleración: {t_lenta/t_rapida:.0f}×')
print()
print('Aplicación real: RSA cifra con exponentes de miles de dígitos.')
print('Sin potencia rápida, cifrar un solo mensaje tomaría años.')

**Mensaje clave:** Dividir el tamaño del problema a la mitad en cada paso es la esencia de D&V.
O(n) → O(log n) es una diferencia de millones cuando n es grande.

### Ejercicio 1 ⭐ — Multiplicación rusa de campesinos

La *multiplicación rusa* calcula `a × b` sin usar multiplicación directa:

```
resultado = 0
mientras b > 0:
    si b es impar:  resultado += a
    a = a × 2          (desplazamiento de bits)
    b = b // 2         (división entera)
```

- **Input:** dos enteros positivos `a`, `b`
- **Output:** `a × b` (entero)
- **Pregunta:** ¿cuál es la complejidad? ¿qué paradigma usa?

In [ ]:
def multiplicacion_rusa(a: int, b: int) -> int:
    '''
    Calcula a × b sin usar el operador *.

    Complejidad:
        Tiempo: O(???)
        Espacio: O(1)
    '''
    raise NotImplementedError('Implementa multiplicacion_rusa')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
casos = [(3, 5, 15), (7, 8, 56), (13, 11, 143), (1, 100, 100), (0, 50, 0)]
ok = True
for a, b, esperado in casos:
    try:
        res = multiplicacion_rusa(a, b)
    except NotImplementedError:
        print('Implementa la función primero.')
        ok = False; break
    if res != esperado:
        print(f'✗ {a} × {b}: esperado {esperado}, obtenido {res}')
        ok = False
    else:
        print(f'✓ {a} × {b} = {res}')
if ok:
    print('\n✅ Correcto. Es O(log b) — idéntica idea que potencia_rapida.')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def multiplicacion_rusa(a, b):
#     resultado = 0
#     while b > 0:
#         if b % 2 == 1:       # b es impar: contribuye a
#             resultado += a
#         a *= 2               # duplicar a
#         b //= 2              # dividir b a la mitad
#     return resultado
#
# Complejidad: O(log b) — el bucle itera log2(b) veces.
# Paradigma: Divide y Vencerás (reduce b a la mitad en cada paso).
print('Solución comentada — descomentar para ver.')

---
## Sección 2 — Greedy
### Problema: Selección de actividades

Dadas n charlas con hora de inicio y fin, selecciona el **máximo número de charlas** que caben en una sola sala.

**Decisión greedy:** siempre elegir la charla que termina más temprano (deja más hueco para las siguientes).

In [ ]:
def actividades_greedy(
        actividades: List[Tuple[int, int, str]]
) -> List[Tuple[int, int, str]]:
    '''
    Selecciona el máximo número de actividades sin solapamiento.
    Criterio greedy: ordenar por tiempo de FIN ascendente.

    Parámetros:
        actividades: lista de (inicio, fin, nombre)

    Complejidad:
        Tiempo: O(n log n)  por el ordenamiento
        Espacio: O(n)
    '''
    ordenadas = sorted(actividades, key=lambda a: a[1]) # ordenamos por fin
    seleccionadas = [] 
    fin_ultimo = -1 

    for inicio, fin, nombre in ordenadas: # iteramos por las actividades ordenadas por fin
        # Solo tomar si no solapa con la última seleccionada
        if inicio >= fin_ultimo:
            seleccionadas.append((inicio, fin, nombre)) # seleccionamos esta actividad
            fin_ultimo = fin

    return seleccionadas


CHARLAS = [
    (1, 4,  'Charla A'),
    (3, 5,  'Charla B'),
    (0, 6,  'Charla C'),
    (5, 7,  'Charla D'),
    (3, 9,  'Charla E'),
    (6, 10, 'Charla F'),
    (8, 11, 'Charla G'),
    (8, 12, 'Charla H'),
    (2, 14, 'Charla I'),
]

seleccion = actividades_greedy(CHARLAS)
print(f'Total charlas: {len(CHARLAS)}')
print(f'Seleccionadas: {len(seleccion)}')
for inicio, fin, nombre in seleccion:
    print(f'  {nombre}: [{inicio}, {fin}]')

In [ ]:
# ── Diagrama de Gantt ───────────────────────────────────────────────────────
nombres_sel = {n for _, _, n in seleccion}

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)

for i, (inicio, fin, nombre) in enumerate(sorted(CHARLAS, key=lambda a: a[1])):
    color = VERDE if nombre in nombres_sel else ROJO
    ax.barh(i, fin - inicio, left=inicio, height=0.6,
            color=color, edgecolor='white', alpha=0.85)
    ax.text((inicio + fin) / 2, i, nombre,
            ha='center', va='center', fontsize=9,
            color='white', fontweight='bold')

ax.set_xlabel('Hora', color=TEXTO, fontsize=11)
ax.set_yticks([])
ax.set_title('Selección de actividades — Greedy (ordenar por fin)',
             fontsize=12, fontweight='bold', color=TEXTO)
ax.tick_params(colors=TEXTO)

leyenda = [
    mpatches.Patch(color=VERDE, label=f'Seleccionada ({len(seleccion)})'),
    mpatches.Patch(color=ROJO,  label=f'Descartada ({len(CHARLAS)-len(seleccion)})'),
]
ax.legend(handles=leyenda, fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

# Demostrar que un criterio incorrecto da peor resultado
def actividades_greedy_duracion(actividades):
    '''Criterio incorrecto: ordenar por duración.'''
    ordenadas = sorted(actividades, key=lambda a: a[1] - a[0])
    seleccionadas, fin_ultimo = [], -1
    for inicio, fin, nombre in ordenadas:
        if inicio >= fin_ultimo:
            seleccionadas.append((inicio, fin, nombre))
            fin_ultimo = fin
    return seleccionadas

sel_dur = actividades_greedy_duracion(CHARLAS)
print(f'Criterio "por fin"     → {len(seleccion)} charlas  ← óptimo')
print(f'Criterio "por duración"→ {len(sel_dur)} charlas  ← subóptimo')
print('\nEl criterio importa: no cualquier greedy funciona.')

**Mensaje clave:** Greedy funciona cuando existe un criterio de selección local que *garantiza* el óptimo global. Aquí, elegir siempre la charla que termina antes deja el mayor hueco posible para las siguientes — esto se puede probar matemáticamente.

### Ejercicio 2 ⭐ — Cambio de monedas con Greedy

Implementa `monedas_greedy(monto, denominaciones)` que devuelva la lista de monedas usadas con la estrategia greedy (siempre usar la moneda más grande que quepa).

- **Input:** `monto: int`, `denominaciones: List[int]` (ordenadas descendente)
- **Output:** `List[int]` con las monedas usadas
- Prueba con `[25, 10, 5, 1]` para monto 41 → debe dar `[25, 10, 5, 1]`
- Prueba con `[4, 3, 1]` para monto 6 → da `[4, 1, 1]` (3 monedas), pero el óptimo es `[3, 3]` (2 monedas)
- **Pregunta:** ¿por qué falla en el segundo caso? Anota tu respuesta como comentario.

In [ ]:
def monedas_greedy(monto: int, denominaciones: List[int]) -> List[int]:
    '''
    Devuelve la lista de monedas usando la estrategia greedy.

    Complejidad:
        Tiempo: O(monto × |denominaciones|)
        Espacio: O(monto)
    '''
    raise NotImplementedError('Implementa monedas_greedy')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
casos = [
    (41,  [25, 10, 5, 1], 41),
    (30,  [25, 10, 5, 1], 30),
    (6,   [4,  3, 1],      6),   # greedy falla → da >2 monedas pero la suma debe ser 6
    (100, [50, 25, 10, 5, 1], 100),
]
ok = True
for monto, dens, esperado_suma in casos:
    try:
        res = monedas_greedy(monto, dens)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    if sum(res) != esperado_suma:
        print(f'✗ monto={monto}: la suma de monedas es {sum(res)}, esperado {esperado_suma}')
        ok = False
    else:
        print(f'✓ monto={monto}, dens={dens} → {res} ({len(res)} monedas)')
if ok:
    print('\n✅ La suma es correcta en todos los casos.')
    print('Nota: para [4,3,1] con monto=6 greedy da 3 monedas; el óptimo son 2 (→ DP en Sección 3).')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def monedas_greedy(monto, denominaciones):
#     dens = sorted(denominaciones, reverse=True)
#     usadas = []
#     for moneda in dens:
#         while monto >= moneda:
#             usadas.append(moneda)
#             monto -= moneda
#     return usadas
#
# Falla con [4,3,1] y monto=6 porque la moneda de 4 parece la mejor
# opción local, pero no permite llegar al óptimo global (3+3=6).
# Greedy no tiene forma de "deshacer" la elección de 4.
print('Solución comentada — descomentar para ver.')

---
## Sección 3 — Programación Dinámica
### Problema: Cambio de monedas mínimo

El mismo problema donde Greedy falla: con denominaciones `[1, 3, 4]` y monto 6, Greedy da 3 monedas (4+1+1), pero el óptimo son 2 (3+3).

**Idea DP:** guardar en una tabla `dp[m]` = mínimo de monedas para el monto `m`. Llenar de 0 a `monto`.

In [ ]:
def cambio_dp(
        monto: int,
        denominaciones: List[int],
        verbose: bool = False
) -> Tuple[int, List[int]]:
    '''
    Cambio mínimo de monedas usando Programación Dinámica.
    dp[m] = mínimo de monedas para el monto m.
    Reconstrucción: seguir qué moneda se usó en cada posición.

    Complejidad:
        Tiempo: O(monto × |denominaciones|)
        Espacio: O(monto)
    '''
    INF = float('inf') # representa un monto imposible de alcanzar
    dp   = [INF] * (monto + 1) # dp[m] = mínimo de monedas para monto m
    usada = [-1]  * (monto + 1)   # qué moneda se usó para llegar a m
    dp[0] = 0 # caso base: 0 monedas para monto 0

    for m in range(1, monto + 1): # iteramos por cada monto desde 1 hasta el monto objetivo
        for moneda in denominaciones:
            if moneda <= m and dp[m - moneda] + 1 < dp[m]: # si podemos usar esta moneda y mejora el resultado
                dp[m]    = dp[m - moneda] + 1
                usada[m] = moneda
        if verbose:
            val = dp[m] if dp[m] < INF else '∞'
            print(f'  dp[{m:2d}] = {val}')

    # Reconstruir cuáles monedas se usaron
    monedas_usadas = []
    m = monto
    while m > 0 and usada[m] != -1:
        monedas_usadas.append(usada[m])
        m -= usada[m]

    return dp[monto], monedas_usadas


# Demostrar el caso donde Greedy fallaba
DENS = [1, 3, 4]
MONTO = 6

n_monedas, monedas = cambio_dp(MONTO, DENS, verbose=True)
print(f'\nResultado: {n_monedas} monedas → {monedas}')
print(f'Greedy daba: 3 monedas → [4, 1, 1]')
print(f'DP da:       {n_monedas} monedas → {sorted(monedas, reverse=True)}  ✓')

In [ ]:
# ── Visualización de la tabla DP ────────────────────────────────────────────
DENS_VIS  = [1, 3, 4]
MONTO_VIS = 10

INF = float('inf')
dp_vis = [INF] * (MONTO_VIS + 1)
usada_vis = [-1] * (MONTO_VIS + 1)
dp_vis[0] = 0
for m in range(1, MONTO_VIS + 1):
    for moneda in DENS_VIS:
        if moneda <= m and dp_vis[m - moneda] + 1 < dp_vis[m]:
            dp_vis[m]    = dp_vis[m - moneda] + 1
            usada_vis[m] = moneda

# Reconstruir la solución para MONTO_VIS
solucion_m = set()
m = MONTO_VIS
while m > 0 and usada_vis[m] != -1:
    solucion_m.add(m)
    m -= usada_vis[m]

fig, ax = plt.subplots(figsize=(12, 3))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)

for i in range(MONTO_VIS + 1):
    val = dp_vis[i]
    if i in solucion_m:
        color = VERDE
    elif i == MONTO_VIS:
        color = NARANJA
    elif val == 0:
        color = MORADO
    else:
        color = AZUL
    ax.bar(i, val if val < INF else 0, color=color, edgecolor='white', width=0.8)
    txt = str(int(val)) if val < INF else '∞'
    ax.text(i, (val if val < INF else 0) + 0.05, txt,
            ha='center', va='bottom', fontsize=11, fontweight='bold', color=TEXTO)
    # Anotación de qué moneda se usó
    if usada_vis[i] != -1:
        ax.text(i, -0.3, f'←{usada_vis[i]}', ha='center', va='top',
                fontsize=8, color='#757575')

ax.set_xticks(range(MONTO_VIS + 1))
ax.set_xticklabels([f'm={i}' for i in range(MONTO_VIS + 1)], fontsize=8, color=TEXTO)
ax.set_ylabel('dp[m] = monedas mínimas', color=TEXTO)
ax.set_title(f'Tabla DP — cambio_dp(monto={MONTO_VIS}, dens={DENS_VIS})\n'
             f'(←x indica qué moneda se usó para llegar a m)',
             fontsize=11, fontweight='bold', color=TEXTO)
ax.set_ylim(-0.6, max(v for v in dp_vis if v < INF) + 0.8)

ley = [
    mpatches.Patch(color=MORADO,  label='Caso base (m=0)'),
    mpatches.Patch(color=AZUL,    label='Calculado'),
    mpatches.Patch(color=VERDE,   label='En la solución óptima'),
    mpatches.Patch(color=NARANJA, label=f'Objetivo m={MONTO_VIS}'),
]
ax.legend(handles=ley, fontsize=9, loc='upper left')
ax.tick_params(colors=TEXTO)
plt.tight_layout()
plt.show()

n_opt, mon_opt = cambio_dp(MONTO_VIS, DENS_VIS)
print(f'Solución óptima para monto={MONTO_VIS}: {n_opt} monedas → {sorted(mon_opt, reverse=True)}')

**Mensaje clave:** La tabla `dp[]` es la *memoria* del algoritmo. En lugar de recalcular el mínimo para cada submonto, lo guardamos y lo reutilizamos. El subproblema "¿cuántas monedas necesito para el monto m?" se solapa para distintas monedas → DP es la herramienta correcta.

### Ejercicio 3 ⭐⭐ — Escaleras

¿De cuántas formas distintas se puede subir una escalera de `n` peldaños si en cada paso se puede subir **1 o 2 peldaños**?

- Ejemplos: `n=1` → 1 forma, `n=2` → 2 formas, `n=4` → 5 formas
- **Input:** `n: int`
- **Output:** número de formas (`int`)
- **Pista:** construye la tabla `dp[0..n]` donde `dp[i]` = formas de llegar al peldaño `i`. ¿Qué patrón reconoces?

In [ ]:
def escaleras_dp(n: int) -> int:
    '''
    Cuenta formas de subir n peldaños tomando 1 o 2 a la vez.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(n)  (o O(1) con la versión optimizada)
    '''
    raise NotImplementedError('Implementa escaleras_dp')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
esperados = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
ok = True
for i, esp in enumerate(esperados):
    try:
        res = escaleras_dp(i)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    if res != esp:
        print(f'✗ n={i}: esperado {esp}, obtenido {res}')
        ok = False
    else:
        print(f'✓ n={i}: {res} formas')
if ok:
    print('\n✅ ¡Correcto! ¿Reconoces la secuencia?')
    print('Son los números de Fibonacci: dp[n] = dp[n-1] + dp[n-2].')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def escaleras_dp(n):
#     if n <= 1:
#         return 1
#     dp = [0] * (n + 1)
#     dp[0] = 1   # 1 forma de estar en el peldaño 0 (sin moverse)
#     dp[1] = 1   # 1 forma de llegar al peldaño 1 (un paso de 1)
#     for i in range(2, n + 1):
#         dp[i] = dp[i-1] + dp[i-2]   # llegar desde i-1 (paso 1) o desde i-2 (paso 2)
#     return dp[n]
#
# La secuencia es Fibonacci. dp[n] = F(n+1).
# Versión O(1) espacio: solo guardar los dos últimos valores.
print('Solución comentada — descomentar para ver.')

---
## Sección 4 — Backtracking
### Problema: Coloración de grafos

Dado un grafo, asignar un color a cada nodo de modo que **ningún par de nodos adyacentes comparta color**, usando a lo más `k` colores.

Es un problema NP-completo: no existe algoritmo polinomial conocido para el caso general. Backtracking lo resuelve exactamente, pero en tiempo exponencial.

In [ ]:
def colorear_grafo(
        grafo: Dict[int, List[int]],
        k: int
) -> Optional[Dict[int, int]]:
    '''
    Colorea el grafo con a lo más k colores (backtracking).
    Retorna el diccionario nodo→color, o None si es imposible.

    Poda: si un color entra en conflicto con un vecino, se descarta.

    Complejidad:
        Tiempo: O(k^n)  peor caso (n = nodos)
        Espacio: O(n)   pila de recursión + asignación
    '''
    n = len(grafo)
    asignacion: Dict[int, int] = {}
    explorados = [0]
    podados    = [0]

    def bt(nodo: int) -> bool:
        if nodo == n:
            return True    # todos los nodos coloreados
        for color in range(k):
            explorados[0] += 1
            # Comprobar si color es válido para este nodo
            conflicto = any(
                asignacion.get(vecino) == color
                for vecino in grafo[nodo]
            )
            if conflicto:
                podados[0] += 1
                continue
            asignacion[nodo] = color
            if bt(nodo + 1):
                return True
            del asignacion[nodo]   # retroceder (backtrack)
        return False

    if bt(0): # nodo 0 es el primer nodo a colorear
        print(f'k={k}: solución encontrada. '
              f'Explorados={explorados[0]}, Podados={podados[0]}')
        return asignacion
    print(f'k={k}: imposible colorear. '
          f'Explorados={explorados[0]}, Podados={podados[0]}')
    return None


# Grafo pentágono (C5): necesita 3 colores
GRAFO_C5 = {0: [1, 4], 1: [0, 2], 2: [1, 3], 3: [2, 4], 4: [3, 0]}

for k_prueba in [4, 3,2,1]:
    resultado = colorear_grafo(GRAFO_C5, k_prueba)

In [ ]:
# ── Visualización del grafo coloreado ───────────────────────────────────────
PALETA_COLORES_NODO = [AZUL, NARANJA, VERDE, MORADO, ROJO]
NOMBRES_COLORES     = ['Azul', 'Naranja', 'Verde', 'Morado', 'Rojo']

asig = colorear_grafo(GRAFO_C5, 3)

# Posiciones del pentágono
pos_c5 = {
    i: (math.cos(2*math.pi*i/5 - math.pi/2),
        math.sin(2*math.pi*i/5 - math.pi/2))
    for i in range(5)
}

fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)
ax.axis('off')
ax.set_title('Coloración del pentágono con 3 colores',
             fontsize=12, fontweight='bold', color=TEXTO)

# Aristas
dibujadas = set()
for u, vecinos in GRAFO_C5.items():
    for v in vecinos:
        if (min(u,v), max(u,v)) not in dibujadas:
            dibujadas.add((min(u,v), max(u,v)))
            x0, y0 = pos_c5[u]
            x1, y1 = pos_c5[v]
            ax.plot([x0, x1], [y0, y1], '-', color='#9E9E9E', lw=2, zorder=1)

# Nodos
for nid, (x, y) in pos_c5.items():
    color_idx = asig[nid] if asig else 0
    color_nodo = PALETA_COLORES_NODO[color_idx]
    circ = plt.Circle((x, y), 0.18, color=color_nodo, zorder=2)
    ax.add_patch(circ)
    ax.text(x, y, str(nid), ha='center', va='center',
            fontsize=14, fontweight='bold', color='white', zorder=3)
    ax.text(x * 1.35, y * 1.35,
            NOMBRES_COLORES[asig[nid]] if asig else '?',
            ha='center', va='center', fontsize=9, color=TEXTO)

ax.set_xlim(-1.7, 1.7)
ax.set_ylim(-1.7, 1.7)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('Ningún par de nodos adyacentes comparte color → coloración válida.')
print('El número cromático del pentágono χ(C5) = 3.')

**Mensaje clave:** Backtracking = fuerza bruta con poda anticipada. Al detectar conflictos temprano se evita explorar ramas inútiles. La poda es la diferencia entre "explorar todo" y "explorar lo necesario". Para grafos grandes el problema sigue siendo exponencial, pero en práctica se recorta mucho el árbol.

### Ejercicio 4 ⭐⭐ — Coloración con conteo

Modifica la función `colorear_grafo` para que en lugar de retornar la primera solución, **cuente todas las coloraciones válidas distintas**.

- **Input:** `grafo: Dict`, `k: int`
- **Output:** `int` (número de coloraciones válidas)
- Para el pentágono con k=3 el resultado es 30.
- **Pista:** en lugar de retornar `True` al llegar al nodo n, incrementa un contador y continúa explorando.

In [ ]:
def contar_coloraciones(grafo: Dict[int, List[int]], k: int) -> int:
    '''
    Cuenta el número de coloraciones válidas del grafo con k colores.

    Complejidad:
        Tiempo: O(k^n)  peor caso
        Espacio: O(n)
    '''
    raise NotImplementedError('Implementa contar_coloraciones')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
# Grafo completo K3 (triángulo): con k=3 colores hay 3!=6 coloraciones válidas
K3 = {0: [1, 2], 1: [0, 2], 2: [0, 1]}
casos = [(K3, 3, 6), (GRAFO_C5, 3, 30), (K3, 2, 0)]
ok = True
for grafo, k, esperado in casos:
    try:
        res = contar_coloraciones(grafo, k)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    nodos = len(grafo)
    status = '✓' if res == esperado else '✗'
    print(f'{status} grafo({nodos} nodos), k={k}: {res} coloraciones (esperado {esperado})')
    if res != esperado:
        ok = False
if ok:
    print('\n✅ Correcto.')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def contar_coloraciones(grafo, k):
#     n = len(grafo)
#     asignacion = {}
#     contador = [0]
#
#     def bt(nodo):
#         if nodo == n:
#             contador[0] += 1   # solución completa: contar y seguir
#             return
#         for color in range(k):
#             conflicto = any(
#                 asignacion.get(v) == color for v in grafo[nodo]
#             )
#             if not conflicto:
#                 asignacion[nodo] = color
#                 bt(nodo + 1)
#                 del asignacion[nodo]
#
#     bt(0)
#     return contador[0]
print('Solución comentada — descomentar para ver.')

---
## Sección 5 — Branch & Bound
### Problema: TSP en grafo pequeño (Problema del Viajante)

Dado un grafo completo de `n` ciudades con costos entre ellas, encontrar el **tour de costo mínimo** que visita todas y regresa al origen.

**Idea B&B:** en cada nodo del árbol de búsqueda, calcular una *cota inferior* del costo mínimo posible desde ahí. Si esa cota ya supera el mejor tour encontrado → podar la rama.

In [ ]:
def cota_tsp(matriz: List[List[int]], ruta: List[int]) -> float:
    '''
    Cota inferior admisible para TSP: suma del costo de la ruta parcial
    más el arco mínimo de salida de cada ciudad no visitada.

    Siempre <= costo real de cualquier extensión → admisible.

    Complejidad:
        Tiempo: O(n²)
        Espacio: O(n)
    '''
    n = len(matriz)
    visitados = set(ruta)

    # Costo de la ruta parcial ya recorrida
    costo_parcial = sum(
        matriz[ruta[i]][ruta[i+1]] for i in range(len(ruta)-1)
    )

    # Para cada ciudad no visitada: sumar su arco de salida mínimo
    cota = float(costo_parcial)
    for ciudad in range(n):
        if ciudad not in visitados:
            min_arco = min(
                matriz[ciudad][j]
                for j in range(n)
                if j != ciudad
            )
            cota += min_arco

    return cota


def tsp_bb(matriz: List[List[int]]):
    '''
    TSP exacto con Branch & Bound (DFS + cota inferior).
    Siempre parte desde la ciudad 0.

    Complejidad:
        Tiempo: O(n!) peor caso, mucho mejor en práctica
        Espacio: O(n)
    '''
    n = len(matriz)
    mejor = [float('inf')]
    mejor_ruta: List[List[int]] = [[]]
    explorados = [0]
    podados    = [0]

    def bb(ruta: List[int], costo: int) -> None:
        explorados[0] += 1

        # Ruta completa: añadir el regreso al origen
        if len(ruta) == n:
            total = costo + matriz[ruta[-1]][0]
            if total < mejor[0]:
                mejor[0] = total
                mejor_ruta[0] = ruta + [0]
            return

        visitados = set(ruta)
        for ciudad in range(n):
            if ciudad in visitados:
                continue
            nueva_ruta = ruta + [ciudad]
            nuevo_costo = costo + matriz[ruta[-1]][ciudad]
            # Poda B&B: la cota ya supera el mejor conocido
            if cota_tsp(matriz, nueva_ruta) >= mejor[0]:
                podados[0] += 1
                continue
            bb(nueva_ruta, nuevo_costo)

    bb([0], 0)
    return mejor[0], mejor_ruta[0], explorados[0], podados[0]


def tsp_bt(matriz: List[List[int]]):
    '''TSP por backtracking puro (sin cota), para comparar.'''
    n = len(matriz)
    mejor = [float('inf')]
    mejor_ruta: List[List[int]] = [[]]
    explorados = [0]

    def bt(ruta, costo):
        explorados[0] += 1
        if len(ruta) == n:
            total = costo + matriz[ruta[-1]][0]
            if total < mejor[0]:
                mejor[0] = total
                mejor_ruta[0] = ruta + [0]
            return
        visitados = set(ruta)
        for ciudad in range(n):
            if ciudad not in visitados:
                bt(ruta + [ciudad], costo + matriz[ruta[-1]][ciudad])

    bt([0], 0)
    return mejor[0], mejor_ruta[0], explorados[0]


# Ejemplo: 5 ciudades, matriz de distancias
MATRIZ_TSP = [
    [  0, 10, 15, 20, 25],
    [ 10,  0, 35, 25, 30],
    [ 15, 35,  0, 30, 10],
    [ 20, 25, 30,  0, 15],
    [ 25, 30, 10, 15,  0],
]

costo_bb, ruta_bb, exp_bb, pod_bb = tsp_bb(MATRIZ_TSP)
costo_bt, ruta_bt, exp_bt         = tsp_bt(MATRIZ_TSP)

print(f'Costo óptimo    : {costo_bb}')
print(f'Ruta óptima     : {" → ".join(map(str, ruta_bb))}')
print()
print(f'{"Métrica":<25} {"Backtracking":>14} {"Branch & Bound":>14}')
print('-' * 55)
print(f'{"Nodos explorados":<25} {exp_bt:>14} {exp_bb:>14}')
print(f'{"Nodos podados":<25} {"—":>14} {pod_bb:>14}')
print(f'{"Reducción":<25} {"—":>14} {(1-exp_bb/exp_bt)*100:>13.1f}%')

In [ ]:
# ── Visualización: ruta óptima sobre el grafo de 5 ciudades ────────────────
# Posiciones de las ciudades en círculo
n_c = len(MATRIZ_TSP)
pos_tsp = {
    i: (math.cos(2*math.pi*i/n_c - math.pi/2),
        math.sin(2*math.pi*i/n_c - math.pi/2))
    for i in range(n_c)
}
nombres_ciudades = ['A', 'B', 'C', 'D', 'E']

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(FONDO)
ax.set_facecolor(FONDO)
ax.axis('off')
ax.set_title(f'TSP — Ruta óptima: {" → ".join(nombres_ciudades[i] for i in ruta_bb)}\n'
             f'Costo total: {costo_bb}',
             fontsize=12, fontweight='bold', color=TEXTO)

# Todas las aristas en gris claro
for i in range(n_c):
    for j in range(i+1, n_c):
        x0, y0 = pos_tsp[i]
        x1, y1 = pos_tsp[j]
        ax.plot([x0, x1], [y0, y1], '-', color='#E0E0E0', lw=1, zorder=1)
        xm, ym = (x0+x1)/2, (y0+y1)/2
        ax.text(xm, ym, str(MATRIZ_TSP[i][j]), ha='center', va='center',
                fontsize=8, color='#9E9E9E', zorder=2)

# Ruta óptima en verde
for idx in range(len(ruta_bb)-1):
    u, v = ruta_bb[idx], ruta_bb[idx+1]
    x0, y0 = pos_tsp[u]
    x1, y1 = pos_tsp[v]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=VERDE, lw=2.5))

# Nodos
for i, (x, y) in pos_tsp.items():
    es_origen = (i == 0)
    color_n = NARANJA if es_origen else AZUL
    circ = plt.Circle((x, y), 0.13, color=color_n, zorder=4)
    ax.add_patch(circ)
    ax.text(x, y, nombres_ciudades[i], ha='center', va='center',
            fontsize=13, fontweight='bold', color='white', zorder=5)

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Mensaje clave:** B&B = Backtracking + función de cota. La cota inferior dice: *"aunque todo salga perfecto desde aquí, el costo mínimo posible es X"*. Si X ya supera la mejor solución conocida, no hay razón para explorar esa rama. La calidad de la cota determina cuánto se reduce el árbol.

### Ejercicio 5 ⭐⭐ — TSP Greedy

Implementa `tsp_greedy(matriz)` — versión greedy del TSP: siempre ir a la **ciudad más cercana no visitada**.

- **Input:** `matriz: List[List[int]]`
- **Output:** `(costo: int, ruta: List[int])`
- Compara el resultado con el óptimo de `tsp_bb`.
- **Pregunta:** ¿puede el Greedy superar el doble del óptimo? Busca un contraejemplo con una matriz de 4 ciudades.

In [ ]:
def tsp_greedy(matriz: List[List[int]]) -> Tuple[int, List[int]]:
    '''
    TSP greedy: siempre visitar la ciudad más cercana no visitada.
    Retorna (costo_total, ruta) donde la ruta empieza y termina en 0.

    Complejidad:
        Tiempo: O(n²)
        Espacio: O(n)
    '''
    raise NotImplementedError('Implementa tsp_greedy')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
try:
    costo_g, ruta_g = tsp_greedy(MATRIZ_TSP)
    n_c = len(MATRIZ_TSP)
    # Verificar que la ruta es válida
    assert len(ruta_g) == n_c + 1, 'La ruta debe tener n+1 elementos (regresa al origen)'
    assert ruta_g[0] == ruta_g[-1] == 0, 'Debe comenzar y terminar en ciudad 0'
    assert set(ruta_g[:-1]) == set(range(n_c)), 'Debe visitar todas las ciudades'
    costo_bb_ref, _, _, _ = tsp_bb(MATRIZ_TSP)
    gap = (costo_g - costo_bb_ref) / costo_bb_ref * 100
    print(f'✓ Ruta válida: {" → ".join(map(str, ruta_g))}')
    print(f'  Costo Greedy : {costo_g}')
    print(f'  Costo óptimo : {costo_bb_ref}')
    print(f'  Gap          : {gap:.1f}%')
    if gap == 0:
        print('En este caso el Greedy encontró el óptimo (no siempre ocurre).')
    else:
        print('El Greedy no fue óptimo en este caso.')
except NotImplementedError:
    print('Implementa la función primero.')
except AssertionError as e:
    print(f'✗ Error en la ruta: {e}')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def tsp_greedy(matriz):
#     n = len(matriz)
#     visitados = [False] * n
#     ruta = [0]
#     visitados[0] = True
#     costo = 0
#     actual = 0
#     for _ in range(n - 1):
#         # Encontrar la ciudad más cercana no visitada
#         mejor_dist = float('inf')
#         mejor_ciudad = -1
#         for j in range(n):
#             if not visitados[j] and matriz[actual][j] < mejor_dist:
#                 mejor_dist = matriz[actual][j]
#                 mejor_ciudad = j
#         ruta.append(mejor_ciudad)
#         visitados[mejor_ciudad] = True
#         costo += mejor_dist
#         actual = mejor_ciudad
#     costo += matriz[actual][0]   # regreso al origen
#     ruta.append(0)
#     return costo, ruta
#
# El Greedy para TSP tiene garantía 2-aprox en grafos métricos,
# pero puede ser arbitrariamente malo en grafos generales.
print('Solución comentada — descomentar para ver.')

---
## Cierre — Los 5 paradigmas en una tabla

In [ ]:
# ── Tabla comparativa final ─────────────────────────────────────────────────
filas = [
    ['Divide y Vencerás', 'Potencia xⁿ',          'O(log n)',   'Sí',       'Subproblemas independientes'],
    ['Greedy',            'Selección actividades', 'O(n log n)', 'A veces',  'Decisión local → global'],
    ['Prog. Dinámica',    'Cambio monedas',         'O(n·k)',     'Sí',       'Subproblemas solapados'],
    ['Backtracking',      'Coloración grafos',      'O(kⁿ)',      'Sí',       'Exploración con poda'],
    ['Branch & Bound',    'TSP pequeño',            'O(n!) / <',  'Sí',       'BT + función de cota'],
]
cols = ['Paradigma', 'Problema (hoy)', 'Complejidad', 'Óptimo', 'Cuándo usarlo']

colores_filas = [AZUL, NARANJA, MORADO, VERDE, ROJO]

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor(FONDO)
ax.axis('off')
ax.set_title('Resumen: paradigmas de diseño de algoritmos — S03',
             fontsize=13, fontweight='bold', color=TEXTO, pad=15)

tabla = ax.table(
    cellText=filas,
    colLabels=cols,
    cellLoc='center',
    loc='center'
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1, 2.0)

# Encabezado
for j in range(len(cols)):
    tabla[(0, j)].set_facecolor(TEXTO)
    tabla[(0, j)].set_text_props(color='white', fontweight='bold')

# Filas con el color del paradigma
for i, color in enumerate(colores_filas):
    for j in range(len(cols)):
        tabla[(i+1, j)].set_facecolor(color)
        tabla[(i+1, j)].set_text_props(color='white', fontweight='bold')

plt.tight_layout()
plt.show()

print('Para profundizar: notebooks NB01–NB05 en la carpeta S03_Diseño_Algoritmos/')